# Marlek ACE-Step Colab Backend

This notebook runs the AI music model on free Google Colab GPU and exposes a small FastAPI service for the local Marlek web UI.

Use Runtime > Change runtime type > T4 GPU before running. The first generation downloads model weights and can take a while.

In [ ]:
!nvidia-smi
!python --version

## Install ACE-Step and tunnel tools

This uses the official ACE-Step GitHub repo and a free Cloudflare quick tunnel. No paid token is required.

In [ ]:
from pathlib import Path

%cd /content
if not Path('/content/ACE-Step').exists():
    !git clone --depth 1 https://github.com/ace-step/ACE-Step.git /content/ACE-Step

%cd /content/ACE-Step
!pip install -q -e .
!pip install -q fastapi uvicorn nest_asyncio pydantic requests python-multipart

if not Path('/usr/local/bin/cloudflared').exists():
    !wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x /usr/local/bin/cloudflared

!cloudflared --version

## Start Marlek API

Copy the printed `https://....trycloudflare.com` URL into the local web app's `Colab API adresi` field.

In [ ]:
import asyncio
import os
import random
import re
import subprocess
import threading
import time
import traceback
import uuid
from pathlib import Path
from typing import Any, Optional

import nest_asyncio
import torch
import uvicorn
from fastapi import FastAPI, Request
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse, JSONResponse
from pydantic import BaseModel, Field

from acestep.pipeline_ace_step import ACEStepPipeline

OUTPUT_DIR = Path('/content/marlek_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR = Path('/content/ace_step_checkpoints')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    cuda_major, _ = torch.cuda.get_device_capability(0)
    if cuda_major >= 8 and torch.cuda.is_bf16_supported():
        DTYPE = 'bfloat16'
        os.environ.pop('ACE_PIPELINE_DTYPE', None)
    else:
        DTYPE = 'float16'
        os.environ['ACE_PIPELINE_DTYPE'] = 'float16'
else:
    DTYPE = 'float32'
    os.environ.pop('ACE_PIPELINE_DTYPE', None)
PIPELINE = None

class GenerateRequest(BaseModel):
    prompt: str = Field(min_length=3, max_length=5000)
    lyrics: str = Field(default='[instrumental]', max_length=4000)
    turkish_pronunciation: str = Field(default='', max_length=1200)
    instrument_palette: str = Field(default='', max_length=2500)
    negative_prompt: str = Field(default='', max_length=1800)
    audio_duration: int = Field(default=30, ge=10, le=120)
    infer_step: int = Field(default=27, ge=10, le=80)
    guidance_scale: float = Field(default=15.0, ge=1.0, le=30.0)
    seed: Optional[int] = None
    scheduler_type: str = 'euler'
    cfg_type: str = 'apg'
    omega_scale: float = 10.0

def compose_generation_prompt(req: GenerateRequest):
    parts = [
        req.prompt,
        req.turkish_pronunciation,
        f'Foreground Turkish instruments: {req.instrument_palette}' if req.instrument_palette else '',
        'Use authentic Turkish/Anatolian instrument timbres. Keep baglama, karaduzen, qanun, ney, kaval/mey, ud, darbuka, bendir, davul, kasik/zil distinct when requested.',
        f'Negative guidance: {req.negative_prompt}' if req.negative_prompt else '',
    ]
    return '. '.join(part.strip().strip('.') for part in parts if part and part.strip())[:5000]

def get_pipeline():
    global PIPELINE
    if PIPELINE is None:
        print(f'Loading ACE-Step on {DEVICE}, dtype={DTYPE}. First load can take several minutes.')
        PIPELINE = ACEStepPipeline(
            checkpoint_dir=str(CHECKPOINT_DIR),
            dtype=DTYPE,
            torch_compile=False,
            cpu_offload=True,
            overlapped_decode=True,
        )
    return PIPELINE

def newest_audio_file():
    candidates = sorted(
        list(OUTPUT_DIR.glob('*.wav')) + list(OUTPUT_DIR.glob('*.mp3')) + list(OUTPUT_DIR.glob('*.flac')),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )
    return candidates[0] if candidates else None

def run_generation(req: GenerateRequest, output_path: Path, seed: int):
    pipe = get_pipeline()
    model_prompt = compose_generation_prompt(req)
    print('Prompt preview:', model_prompt[:1000])
    print('Lyrics preview:', req.lyrics[:600])
    result = pipe(
        audio_duration=req.audio_duration,
        prompt=model_prompt,
        lyrics=req.lyrics,
        infer_step=req.infer_step,
        guidance_scale=req.guidance_scale,
        scheduler_type=req.scheduler_type,
        cfg_type=req.cfg_type,
        omega_scale=req.omega_scale,
        manual_seeds=[seed],
        guidance_interval=0.5,
        guidance_interval_decay=0,
        min_guidance_scale=3,
        use_erg_tag=True,
        use_erg_lyric=True,
        use_erg_diffusion=True,
        oss_steps=None,
        guidance_scale_text=0,
        guidance_scale_lyric=0,
        save_path=str(output_path),
    )
    if output_path.exists():
        return output_path
    if isinstance(result, (list, tuple)) and result:
        maybe = Path(str(result[0]))
        if maybe.exists():
            return maybe
    if isinstance(result, str):
        maybe = Path(result)
        if maybe.exists():
            return maybe
    latest = newest_audio_file()
    if latest is not None:
        return latest
    raise RuntimeError('ACE-Step finished but no audio file was found.')

app = FastAPI(title='Marlek ACE-Step Colab API')
app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_credentials=False,
    allow_methods=['*'],
    allow_headers=['*'],
)

@app.get('/health')
def health():
    return {
        'ok': True,
        'model': 'ACE-Step',
        'device': DEVICE,
        'dtype': DTYPE,
        'cuda_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    }

@app.get('/audio/{filename}', name='download_audio')
def download_audio(filename: str):
    safe_name = Path(filename).name
    path = OUTPUT_DIR / safe_name
    if not path.exists():
        return JSONResponse({'error': 'file not found'}, status_code=404)
    return FileResponse(path, media_type='audio/wav', filename=safe_name)

@app.post('/generate')
async def generate(req: GenerateRequest, request: Request):
    seed = int(req.seed or random.randint(1, 2_147_483_000))
    filename = f'marlek_ace_step_{seed}_{uuid.uuid4().hex[:8]}.wav'
    output_path = OUTPUT_DIR / filename
    started = time.time()
    try:
        final_path = await asyncio.to_thread(run_generation, req, output_path, seed)
    except Exception as exc:
        return JSONResponse({'status': 'error', 'error': f'{type(exc).__name__}: {exc}', 'traceback': traceback.format_exc()[-4000:]}, status_code=500)
    elapsed = round(time.time() - started, 2)
    audio_url = str(request.url_for('download_audio', filename=final_path.name))
    return {
        'status': 'ok',
        'model': 'ACE-Step',
        'seed': seed,
        'seconds': req.audio_duration,
        'elapsed_seconds': elapsed,
        'filename': final_path.name,
        'audio_url': audio_url,
    }

nest_asyncio.apply()
server = uvicorn.Server(uvicorn.Config(app, host='0.0.0.0', port=8000, log_level='info'))
threading.Thread(target=server.run, daemon=True).start()
time.sleep(2)
print('Local API: http://127.0.0.1:8000/health')

def start_cloudflared_tunnel():
    proc = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    public_url = None
    deadline = time.time() + 60
    while time.time() < deadline:
        line = proc.stdout.readline()
        if line:
            print(line.rstrip())
            match = re.search(r'https://[-a-zA-Z0-9.]+\.trycloudflare\.com', line)
            if match:
                public_url = match.group(0)
                break
        if proc.poll() is not None:
            raise RuntimeError('Cloudflare tunnel process exited before URL was ready.')
    if not public_url:
        proc.terminate()
        raise RuntimeError('Cloudflare tunnel URL not found. Re-run this cell.')

    def drain_cloudflared_output():
        while True:
            line = proc.stdout.readline()
            if line:
                print(line.rstrip())
            elif proc.poll() is not None:
                print('cloudflared process exited with code', proc.returncode)
                break
            else:
                time.sleep(0.2)

    threading.Thread(target=drain_cloudflared_output, daemon=True).start()
    return proc, public_url

cloudflared, PUBLIC_URL = start_cloudflared_tunnel()

print('\nCOPY THIS INTO THE LOCAL APP:')
print(PUBLIC_URL)
print('\nHealth check:', PUBLIC_URL + '/health')

## Browser-safe async generation endpoints

These endpoints avoid browser and tunnel timeouts by starting generation in the background and polling job status from the local app.

In [ ]:
JOBS = {}
JOBS_LOCK = threading.Lock()
GPU_GENERATION_LOCK = threading.Lock()
JOB_TTL_SECONDS = 6 * 60 * 60

def cleanup_jobs():
    now = time.time()
    with JOBS_LOCK:
        stale_ids = [
            job_id
            for job_id, job in JOBS.items()
            if now - job.get('updated_at', now) > JOB_TTL_SECONDS and job.get('status') in {'ok', 'error', 'failed'}
        ]
        for job_id in stale_ids:
            JOBS.pop(job_id, None)

def update_job(job_id, **updates):
    with JOBS_LOCK:
        job = JOBS.get(job_id)
        if not job:
            return None
        job.update(updates)
        job['updated_at'] = time.time()
        return dict(job)

def job_payload(job_id, request: Request):
    with JOBS_LOCK:
        job = JOBS.get(job_id)
        if not job:
            return None
        payload = dict(job)
    payload['job_id'] = job_id
    payload['status_url'] = str(request.url_for('get_job', job_id=job_id))
    if payload.get('filename'):
        payload['audio_url'] = str(request.url_for('download_audio', filename=payload['filename']))
    return payload

def run_async_generation_job(job_id, req: GenerateRequest, output_path: Path, seed: int):
    update_job(job_id, status='queued', message='Waiting for the free Colab GPU slot')
    started = time.time()
    try:
        with GPU_GENERATION_LOCK:
            update_job(job_id, status='running', message='Loading ACE-Step and generating audio')
            final_path = run_generation(req, output_path, seed)
        update_job(
            job_id,
            status='ok',
            message='Generation complete',
            filename=final_path.name,
            elapsed_seconds=round(time.time() - started, 2),
        )
    except Exception as exc:
        error_detail = traceback.format_exc()[-4000:]
        update_job(
            job_id,
            status='error',
            message='Generation failed',
            error=f'{type(exc).__name__}: {exc}',
            traceback=error_detail,
            elapsed_seconds=round(time.time() - started, 2),
        )

@app.post('/generate_async')
async def generate_async(req: GenerateRequest, request: Request):
    cleanup_jobs()
    seed = int(req.seed or random.randint(1, 2_147_483_000))
    filename = f'marlek_ace_step_{seed}_{uuid.uuid4().hex[:8]}.wav'
    output_path = OUTPUT_DIR / filename
    job_id = uuid.uuid4().hex
    now = time.time()
    with JOBS_LOCK:
        JOBS[job_id] = {
            'status': 'queued',
            'message': 'Generation queued',
            'model': 'ACE-Step',
            'seed': seed,
            'seconds': req.audio_duration,
            'filename': None,
            'created_at': now,
            'updated_at': now,
        }
    threading.Thread(target=run_async_generation_job, args=(job_id, req, output_path, seed), daemon=True).start()
    return job_payload(job_id, request)

@app.get('/jobs/{job_id}', name='get_job')
def get_job(job_id: str, request: Request):
    payload = job_payload(job_id, request)
    if not payload:
        return JSONResponse({'status': 'error', 'error': 'job not found'}, status_code=404)
    return payload

print('Async API ready:', PUBLIC_URL + '/generate_async')
print('Job status URL format:', PUBLIC_URL + '/jobs/<job_id>')

## Optional 15 second quality test inside Colab

Run this cell after the API URL appears. It generates one short bozlak-style clip and plays it inside Colab.

In [ ]:
import requests
import time
from IPython.display import Audio, display

test_payload = {
    'prompt': 'Kırşehir bozlak, karadüzen bağlama, bağlama saz lead, ney nefesi, kanun arpejleri, kaval/mey rengi, ud sıcaklığı, darbuka ve bendir dokusu, duygulu Türkçe erkek folk vokal, clean mix, no direct artist voice clone',
    'lyrics': '[verse]\nBozkırın yelinde kaldım bir gece\nSazımın telinde inledi hece\nYol uzun gurbetin sesi derinde\nBir ah çeker gönlüm dağlar içinde',
    'turkish_pronunciation': 'Vocal language is Turkish. Read dotless ı as Turkish ı, not English i. Pronounce ş, ç, ğ, ö, ü naturally. Do not anglicize the lyrics.',
    'instrument_palette': 'bağlama / saz; bozuk düzen / karadüzen bağlama; elektro bağlama; kanun / qanun; ney; kaval / mey; ud / oud; darbuka; bendir; davul; kaşık / zil',
    'negative_prompt': 'avoid English pronunciation, avoid generic guitar instead of bağlama, avoid piano instead of kanun, avoid generic flute instead of ney',
    'audio_duration': 15,
    'infer_step': 27,
    'guidance_scale': 15,
    'seed': 42842,
}

response = requests.post(PUBLIC_URL + '/generate_async', json=test_payload, timeout=120)
print(response.status_code)
print(response.text[:1000])
response.raise_for_status()
job = response.json()
audio_url = None
for attempt in range(360):
    status_response = requests.get(job['status_url'], timeout=120)
    status_response.raise_for_status()
    status = status_response.json()
    print(attempt, status.get('status'), status.get('message'), status.get('elapsed_seconds'))
    if status.get('status') in {'ok', 'done', 'completed'}:
        audio_url = status['audio_url']
        break
    if status.get('status') in {'error', 'failed'}:
        raise RuntimeError(status.get('error') or 'Generation failed')
    time.sleep(5)
if not audio_url:
    raise TimeoutError('Generation did not finish in time')
audio_bytes = requests.get(audio_url, timeout=120).content
display(Audio(audio_bytes, autoplay=False))